<a href="https://colab.research.google.com/github/soumnemishra/agentic_agriculture/blob/agentic/ACA_MODEL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# mounting the google drive

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ==========================
# Standard Library
# ==========================
import os

# ==========================
# TensorFlow
# ==========================
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras import Model
from tensorflow.keras import layers
from tensorflow.keras.layers import (
    Conv2D,
    Dense,
    MaxPooling2D,
    GlobalAveragePooling2D,
    Reshape,
    concatenate,
    multiply
)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:

tomato_dir=r"/content/drive/MyDrive/plant_disease_dataset/data_/Tomato_"
print(f"Checking contents of: {tomato_dir}")

# List contents of the base dataset path (e.g., train, validation, test folders)
if os.path.exists(tomato_dir):
    print(f"Contents found at {tomato_dir}:")
    for item in os.listdir(tomato_dir):
        item_path = os.path.join(tomato_dir, item)
        print(f"- {item} {'(Directory)' if os.path.isdir(item_path) else '(File)'}")
else:
    print(f"Base dataset path does not exist: {tomato_dir}")


Checking contents of: /content/drive/MyDrive/plant_disease_dataset/data_/Tomato_
Contents found at /content/drive/MyDrive/plant_disease_dataset/data_/Tomato_:
- train (Directory)
- valid (Directory)


In [ ]:
train_dir = os.path.join(tomato_dir, 'train') # Assuming 'train' is the directory with class folders


print(f"Folders in '{train_dir}':")

# List all entries in the train_dir
for item in os.listdir(train_dir):
    item_path = os.path.join(train_dir, item)
    # Check if the item is a directory (a class folder)
    if os.path.isdir(item_path):
        print(f"- {item}")

Folders in '/content/drive/MyDrive/plant_disease_dataset/data_/Tomato_/train':
- Septoria_leaf_spot
- powdery_mildew
- Tomato_mosaic_virus
- Bacterial_spot
- Early_blight
- Late_blight
- Leaf_Mold
- Spider_mites Two-spotted_spider_mite
- Tomato_Yellow_Leaf_Curl_Virus
- healthy
- Target_Spot


In [ ]:



if os.path.exists(train_dir):
    print(f"\nCounting files in each class directory within: {train_dir}")
    class_counts = {}
    for class_name in os.listdir(train_dir):
        class_path = os.path.join(train_dir, class_name)
        if os.path.isdir(class_path):
            # Count only image files (e.g., .jpg, .jpeg, .png)
            num_files = len([name for name in os.listdir(class_path) if name.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp'))])
            class_counts[class_name] = num_files

    # Sort for consistent output
    sorted_class_counts = sorted(class_counts.items())

    for class_name, count in sorted_class_counts:
        print(f"{class_name}: {count} files")
else:
    print(f"The directory '{train_dir}' does not exist. Please check the dataset structure.")


Counting files in each class directory within: /content/drive/MyDrive/plant_disease_dataset/data_/Tomato_/train
Bacterial_spot: 2826 files
Early_blight: 2455 files
Late_blight: 3113 files
Leaf_Mold: 2754 files
Septoria_leaf_spot: 2882 files
Spider_mites Two-spotted_spider_mite: 1747 files
Target_Spot: 1827 files
Tomato_Yellow_Leaf_Curl_Virus: 2039 files
Tomato_mosaic_virus: 2153 files
healthy: 3051 files
powdery_mildew: 1004 files


In [ ]:

# this is for the  data integrity check to match with the original data set
def count_files_and_extensions(directory):
    total_files = 0
    extension_counts = {}

    for root, _, files in os.walk(directory):
        for file in files:
            total_files += 1
            ext = os.path.splitext(file)[1].lower()
            extension_counts[ext] = extension_counts.get(ext, 0) + 1

    return total_files, extension_counts

# Assuming base_dataset_path is already defined from previous cells
# base_dataset_path = '/content/drive/MyDrive/plant_disease_dataset/data_/Tomato_'

if 'tomato_dir' in locals() and os.path.exists(tomato_dir):
    total, extensions = count_files_and_extensions(tomato_dir)
    print(f"Total files in dataset: {total}")
    print("File extension breakdown:")
    for ext, count in sorted(extensions.items()):
        print(f"  {ext}: {count}")
else:
    print(f"Base dataset path '{tomato_dir}' not found or not defined.")

Total files in dataset: 32534
File extension breakdown:
  .jpeg: 4
  .jpg: 32025
  .png: 505


#Transfering files from the Drive dir to the Local train dir and validation_dir

# this code is for the collab run time copy of the files


In [ ]:
import shutil
import os
import tensorflow as tf

# 1. Define paths
drive_train_dir = '/content/drive/MyDrive/plant_disease_dataset/data_/Tomato_/train'
drive_valid_dir = '/content/drive/MyDrive/plant_disease_dataset/data_/Tomato_/valid'

local_train_dir = '/content/local_data/train'
local_valid_dir = '/content/local_data/valid'

# 2. Function to synchronize data from Drive to Local Runtime
def sync_local_data(src, dst):
    if os.path.exists(src):
        # Add a cleanup step: remove existing local directory if it exists
        if os.path.exists(dst):
            print(f"Removing existing local data at {dst}...")
            shutil.rmtree(dst)

        print(f"Copying {src} to local runtime...")
        shutil.copytree(src, dst)
    else:
        print(f"Source directory does not exist: {src}")

# Synchronize training data
sync_local_data(drive_train_dir, local_train_dir)

# Synchronize validation data
sync_local_data(drive_valid_dir, local_valid_dir)

Removing existing local data at /content/local_data/train...
Copying /content/drive/MyDrive/plant_disease_dataset/data_/Tomato_/train to local runtime...
Removing existing local data at /content/local_data/valid...
Copying /content/drive/MyDrive/plant_disease_dataset/data_/Tomato_/valid to local runtime...


In [ ]:
# we are intiating the Dense121 , a highly efficient convolutional neural network
#known for its connecting layer to every other layer

###  include_top=False,
#we chopped out the top layer don't want the model to output standard
#ImageNet classes (like "dog" or "car"); you want raw extracted features so you can add your own custom layers (like CondConv) on top.

 # weights="imagenet",
 #Instead of starting with random weights, the model initializes with weights learned from millions of images on the ImageNet dataset.
base_model = tf.keras.applications.MobileNetV2(
    include_top=False,
    weights="imagenet",
    input_shape=(224,224,3)
)

base_model.trainable = False

In [ ]:
# ----------------------------
# DATASET LOADING
# ----------------------------

batch_size = 32
img_height = 224
img_width = 224

print("Loading Training Dataset:")

train_ds = tf.keras.utils.image_dataset_from_directory(
    r'/content/local_data/train',
    image_size=(img_height, img_width),
    batch_size=batch_size,
    label_mode='categorical',
    shuffle=True
)

print("\nLoading Validation Dataset:")

val_ds = tf.keras.utils.image_dataset_from_directory(
    r'/content/local_data/valid',
    image_size=(img_height, img_width),
    batch_size=batch_size,
    label_mode='categorical',
    shuffle=False
)

# ----------------------------
# PERFORMANCE SETTINGS
# ----------------------------

AUTOTUNE = tf.data.AUTOTUNE

preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input

# ----------------------------
# PREPROCESS IMAGES
# ----------------------------

train_ds = train_ds.map(
    lambda x, y: (preprocess_input(x), y),
    num_parallel_calls=AUTOTUNE
)

val_ds = val_ds.map(
    lambda x, y: (preprocess_input(x), y),
    num_parallel_calls=AUTOTUNE
)

# ----------------------------
# PIPELINE OPTIMIZATION
# ----------------------------

# train_ds = train_ds.shuffle(1000).prefetch(AUTOTUNE)

# val_ds = val_ds.prefetch(AUTOTUNE)

train_ds = train_ds.prefetch(buffer_size=2)
val_ds = val_ds.prefetch(buffer_size=2)

Loading Training Dataset:
Found 25851 files belonging to 11 classes.

Loading Validation Dataset:
Found 6683 files belonging to 11 classes.


In [ ]:
# =====================================
# DEVELOPMENT DATASET
# =====================================

# Use only 50 training batches
small_train_ds = train_ds.take(50)

# Use only 10 validation batches
small_val_ds = val_ds.take(10)

print("Development dataset created.")

Development dataset created.


In [ ]:
print("Training batches:", tf.data.experimental.cardinality(small_train_ds).numpy())
print("Validation batches:", tf.data.experimental.cardinality(small_val_ds).numpy())

Training batches: 50
Validation batches: 10


In [ ]:
for images, labels in small_train_ds.take(1):

    print("Image batch shape :", images.shape)
    print("Label batch shape :", labels.shape)

    break

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

class_names = train_ds.class_names

plt.figure(figsize=(12,12))

for images, labels in small_train_ds.take(1):

    for i in range(9):

        plt.subplot(3,3,i+1)

        img = images[i].numpy()

        # Undo MobileNetV2 preprocessing
        img = (img + 1.0) * 127.5
        img = np.clip(img,0,255).astype(np.uint8)

        plt.imshow(img)

        plt.title(class_names[np.argmax(labels[i])])

        plt.axis("off")

plt.show()

In [ ]:
WEIGHT_DECAY = 2e-4

def conv2d(kernel_size, stride, filters, kernel_regularizer=tf.keras.regularizers.l2(WEIGHT_DECAY), padding="same", use_bias=False,
           kernel_initializer="he_normal", **kwargs):
    return layers.Conv2D(kernel_size=kernel_size, strides=stride, filters=filters, kernel_regularizer=kernel_regularizer, padding=padding,
                         use_bias=use_bias, kernel_initializer=kernel_initializer, **kwargs)

class Routing(layers.Layer):
    def __init__(self, out_channels, dropout_rate, temperature=30, **kwargs):
        super(Routing, self).__init__(**kwargs)
        self.avgpool = layers.GlobalAveragePooling2D()
        self.dropout = layers.Dropout(rate=dropout_rate)
        self.fc = layers.Dense(units=out_channels)
        self.softmax = layers.Softmax()
        self.temperature = temperature

    def call(self, inputs, **kwargs):
        out = self.avgpool(inputs)
        out = self.dropout(out)
        out = self.softmax(self.fc(out) * 1.0 / self.temperature)
        return out

class CondConv2D(layers.Layer):
    def __init__(self, filters, kernel_size, stride=1, use_bias=True, num_experts=3, padding="same", **kwargs):
        super(CondConv2D, self).__init__(**kwargs)
        self.routing = Routing(out_channels=num_experts, dropout_rate=0.2, name="routing_layer")
        self.convs = []
        for _ in range(num_experts):
            self.convs.append(conv2d(filters=filters, stride=stride, kernel_size=kernel_size, use_bias=use_bias, padding=padding))

    def call(self, inputs, **kwargs):
        routing_weights = self.routing(inputs)

        # MEMORY FIX: Reshape and broadcast instead of massive tf.transpose operations
        weight_0 = tf.reshape(routing_weights[:, 0], (-1, 1, 1, 1))
        feature = weight_0 * self.convs[0](inputs)

        for i in range(1, len(self.convs)):
            weight_i = tf.reshape(routing_weights[:, i], (-1, 1, 1, 1))
            feature += weight_i * self.convs[i](inputs)

        return feature

In [ ]:
kernel_init = tf.keras.initializers.glorot_uniform()
bias_init = tf.keras.initializers.Constant(value=0.0)

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.2),
    # Gaussian noise simulates the graininess of a fast-moving drone camera
    layers.GaussianNoise(0.1)
], name="drone_condition_augmentation")

In [ ]:
def Inception(x, nb_filter):
    # Removed the hardcoded name="..." arguments so Keras auto-names them safely
    branch1x1 = CondConv2D(kernel_size=(1,1), filters=nb_filter, stride=1, padding='same', use_bias=True,  num_experts=3)(x)

    branch3x3 = CondConv2D(kernel_size=(1,1), filters=nb_filter, stride=1, padding='same', use_bias=True,  num_experts=3)(x)
    branch3x31 = CondConv2D(kernel_size=(3,1), filters=nb_filter, stride=1, padding='same', use_bias=True,  num_experts=3)(branch3x3)
    branch3x32 = CondConv2D(kernel_size=(1,3), filters=nb_filter, stride=1, padding='same', use_bias=True,  num_experts=3)(branch3x3)
    out1 = layers.Add()([branch3x31, branch3x32])

    branch5x5 = CondConv2D(kernel_size=(1,1), filters=nb_filter, stride=1, padding='same', use_bias=True,  num_experts=3)(x)
    branch5x5_1 = CondConv2D(kernel_size=(3,1), filters=nb_filter, stride=1, padding='same', use_bias=True,  num_experts=3)(branch5x5)
    branch5x5_2 = CondConv2D(kernel_size=(1,3), filters=nb_filter, stride=1, padding='same', use_bias=True,  num_experts=3)(branch5x5)
    out2 = layers.Add()([branch5x5_1, branch5x5_2])

    branch5x51 = CondConv2D(kernel_size=(3,1), filters=nb_filter, stride=1, padding='same', use_bias=True,  num_experts=3)(out2)
    branch5x52 = CondConv2D(kernel_size=(1,3), filters=nb_filter, stride=1, padding='same', use_bias=True,  num_experts=3)(out2)
    out3 = layers.Add()([branch5x51, branch5x52])

    branchpool = MaxPooling2D(pool_size=(3,3), strides=(1,1), padding='same')(x)
    branchpool = CondConv2D(kernel_size=(1,1), filters=nb_filter, stride=1, padding='same', use_bias=True,  num_experts=3)(branchpool)

    # Concatenate all paths together
    x = concatenate([branch1x1, out1, out3, branchpool], axis=3)
    return x

In [ ]:
def mlp(x, hidden_units, dropout_rate):
    for units in hidden_units:
        x = layers.Dense(units, activation=tf.nn.gelu)(x)
        x = layers.Dropout(dropout_rate)(x)
    return x

In [ ]:
class Patches(layers.Layer):
    def __init__(self, patch_size, **kwargs):
        super(Patches, self).__init__(**kwargs)
        self.patch_size = patch_size

    def get_config(self):
        config = super().get_config()
        config.update({
            "patch_size": self.patch_size
        })
        return config

    def call(self, images):
        # Print the input feature map shape
        print("=" * 60)
        print("INPUT TO PATCHES :", images.shape)

        batch_size = tf.shape(images)[0]

        patches = tf.image.extract_patches(
            images=images,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates=[1, 1, 1, 1],
            padding="VALID",
        )

        print("PATCH TENSOR :", patches.shape)

        patch_dims = patches.shape[-1]

        patches = tf.reshape(
            patches,
            [batch_size, -1, patch_dims]
        )

        print("PATCHES AFTER RESHAPE :", patches.shape)
        print("=" * 60)

        return patches

In [ ]:
class PatchEncoder(layers.Layer):
    def __init__(self, num_patches, projection_dim, **kwargs):
        super(PatchEncoder, self).__init__(**kwargs)

        self.num_patches = num_patches
        self.projection_dim = projection_dim

        self.projection = layers.Dense(projection_dim)
        self.position_embedding = layers.Embedding(
            input_dim=num_patches,
            output_dim=projection_dim
        )

    def call(self, patch):

        print("=" * 60)
        print("INPUT PATCHES :", patch.shape)

        positions = tf.range(
            start=0,
            limit=self.num_patches,
            delta=1
        )

        print("NUMBER OF POSITIONS :", positions.shape)

        projected = self.projection(patch)

        print("PROJECTED PATCHES :", projected.shape)

        embedded = self.position_embedding(positions)

        print("POSITION EMBEDDING :", embedded.shape)

        output = projected + embedded

        print("FINAL OUTPUT :", output.shape)
        print("=" * 60)

        return output

In [ ]:
def sse_block(input_feature, ratio=4):
    """Implementation of Squeeze-and-Excitation(SE) block using Keras Layers."""
    channel_axis = -1
    channel = input_feature.shape[channel_axis]

    # Squeeze & Excitation Path
    se_feature = layers.GlobalAveragePooling2D()(input_feature)
    se_feature = layers.Reshape((1, 1, channel))(se_feature)
    se_feature = layers.Dense(channel // ratio, activation='relu', kernel_initializer='he_normal')(se_feature)
    se_feature = layers.Dense(channel, activation='sigmoid', kernel_initializer='he_normal')(se_feature)

    # Multiply the input by the SE features
    se_out = layers.Multiply()([input_feature, se_feature])

    # Spatial Statistics (Mean, Std, Max)
    # We wrap tf functions in Lambda layers to avoid the KerasTensor error
    mean = layers.Lambda(lambda x: tf.reduce_mean(x, axis=-1, keepdims=True))(input_feature)
    std = layers.Lambda(lambda x: tf.math.reduce_std(x, axis=-1, keepdims=True))(input_feature)
    maximum = layers.Lambda(lambda x: tf.reduce_max(x, axis=-1, keepdims=True))(input_feature)

    # Final Concatenation
    out = layers.Concatenate()([se_out, mean, std, maximum])

    return out

In [ ]:
# image_size =56
# patch_size = 5  # Size of the patches to be extract from the input images
# num_patches = (image_size // patch_size) ** 2
# projection_dim = 32
# num_heads = 4
# transformer_units = [
#     projection_dim * 2,
#     projection_dim,
# ]  # Size of the transformer layers
# transformer_layers = 4


In [ ]:
# --- 1. SET UP HYPERPARAMETERS ---
image_size = 55  # CHANGED: Increased from 28 to 55 to force exactly 121 patches
patch_size = 5
num_patches = (image_size // patch_size) ** 2 # This now correctly evaluates to 121
projection_dim = 32
num_heads = 2
transformer_units = [projection_dim * 2, projection_dim]
transformer_layers = 2
TOTAL_CLASSES = 11

# --- 2. AUGMENTED GRAPH TRICK ---
inputs = tf.keras.Input(shape=(224, 224, 3))
augmented_x = data_augmentation(inputs)

base_model = tf.keras.applications.MobileNetV2(
    include_top=False,
    weights="imagenet",
    input_tensor=augmented_x
)
base_model.trainable = False

# --- 3. MULTI-SCALE FEATURE EXTRACTION ---
# Branch 1: High-level features. Output from base is 112x112.
x1 = base_model.get_layer('expanded_conv_project_BN').output
x1 = CondConv2D(kernel_size=3, filters=16, stride=2, padding='same', num_experts=3)(x1)
x1 = CondConv2D(kernel_size=3, filters=32, stride=2, padding='same', num_experts=3)(x1)
x1 = CondConv2D(kernel_size=3, filters=32, stride=2, padding='same', num_experts=3)(x1)
x1 = Conv2D(kernel_size=4, filters=29, strides=1, padding='valid')(x1)
x_conv1 = sse_block(x1, ratio=4)

# Branch 2: Mid-level features. Output from base is 56x56.
x2 = base_model.get_layer('block_2_project_BN').output
x2 = CondConv2D(kernel_size=3, filters=16, stride=2, padding='same', num_experts=3)(x2)
x2 = CondConv2D(kernel_size=3, filters=32, stride=2, padding='same', num_experts=3)(x2)
x2 = Conv2D(kernel_size=4, filters=29, strides=1, padding='valid')(x2)
x_conv2 = sse_block(x2, ratio=4)

# Branch 3: Deep features + Inception. Output from base is 28x28.
x_inc = base_model.get_layer('block_5_add').output
x_inc_out = Inception(x_inc, 32)
x3 = CondConv2D(kernel_size=3, filters=32, stride=2, padding='same', num_experts=3)(x_inc_out)
x3 = Conv2D(kernel_size=4, filters=29, strides=1, padding='valid')(x3)
x_conv3 = sse_block(x3, ratio=4)

# --- 4. VISION TRANSFORMER PATH ---
# ADDED: Resize the feature map so patch extraction yields exactly 121 patches
x_inc_resized = layers.Resizing(image_size, image_size)(x_inc_out)

patches = Patches(patch_size)(x_inc_resized)
encoded_patches = PatchEncoder(num_patches, projection_dim)(patches)
encoded_patches = layers.Dropout(0.1)(encoded_patches)

for _ in range(transformer_layers):
    x_norm1 = layers.LayerNormalization(epsilon=1e-6)(encoded_patches)
    attention_output = layers.MultiHeadAttention(num_heads=num_heads, key_dim=projection_dim, dropout=0.1)(x_norm1, x_norm1)
    x_add1 = layers.Add()([attention_output, encoded_patches])
    x_norm2 = layers.LayerNormalization(epsilon=1e-6)(x_add1)
    x_mlp = mlp(x_norm2, hidden_units=transformer_units, dropout_rate=0.1)
    encoded_patches = layers.Add()([x_mlp, x_add1])

x_final_trans = layers.LayerNormalization(epsilon=1e-6, name='cam_layer')(encoded_patches)
x_t = layers.Reshape((11, 11, 32))(x_final_trans) # This reshape will now work perfectly.

# --- 5. FUSION & CLASSIFICATION ---
# Now all inputs to Add() are exactly (None, 11, 11, 32)
merged = layers.Add()([x_conv1, x_conv2, x_conv3, x_t])
merged = sse_block(merged, ratio=4)
merged = layers.GlobalAveragePooling2D()(merged)
predictions = Dense(TOTAL_CLASSES, activation='softmax')(merged)

drone_model = Model(inputs=inputs, outputs=predictions)
drone_model.summary()

/tmp/ipykernel_41739/2170716328.py:15: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = tf.keras.applications.MobileNetV2(


INPUT TO PATCHES : (None, 55, 55, 128)
PATCH TENSOR : (None, 11, 11, 3200)
PATCHES AFTER RESHAPE : (None, None, 3200)
INPUT PATCHES : (None, None, 3200)
NUMBER OF POSITIONS : (121,)
PROJECTED PATCHES : (None, None, 32)
POSITION EMBEDDING : (121, 32)
FINAL OUTPUT : (None, 121, 32)
INPUT PATCHES : (None, None, 3200)
NUMBER OF POSITIONS : (121,)
PROJECTED PATCHES : (None, None, 32)
POSITION EMBEDDING : (121, 32)
FINAL OUTPUT : (None, 121, 32)


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ drone_condition_au… │ (None, 224, 224,  │          0 │ input_layer_1[0]… │
│ (Sequential)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ drone_condition_… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis

 Total params: 491,649 (1.88 MB)

 Trainable params: 432,641 (1.65 MB)

 Non-trainable params: 59,008 (230.50 KB)

In [ ]:
import numpy as np
import os
from sklearn.utils.class_weight import compute_class_weight

print("="*60)
print("COMPUTING CLASS WEIGHTS TO COMBAT IMBALANCE")
print("="*60)

# FIX: Get class names directly from the folder structure since train_ds lost the attribute
# We use sorted() to ensure the order exactly matches how Keras loads them
class_names = sorted([d for d in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, d))])

# 2. Count files per class in the training directory
class_counts = []
for current_class in class_names:
    class_dir = os.path.join(train_dir, current_class)
    # Count images in this specific folder
    num_files = len([name for name in os.listdir(class_dir) if name.lower().endswith(('.png', '.jpg', '.jpeg'))])
    class_counts.append(num_files)

# 3. Create a mock array of labels to feed into sklearn
y_train_mock = []
for i, count in enumerate(class_counts):
    y_train_mock.extend([i] * count)

# 4. Compute the weights
weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_mock),
    y=y_train_mock
)

# 5. Format into a dictionary for Keras
class_weight_dict = {i: weight for i, weight in enumerate(weights)}

for i, class_name in enumerate(class_names):
    print(f"{class_name}: Weight = {class_weight_dict[i]:.4f}")

COMPUTING CLASS WEIGHTS TO COMBAT IMBALANCE
Bacterial_spot: Weight = 0.8316
Early_blight: Weight = 0.9573
Late_blight: Weight = 0.7549
Leaf_Mold: Weight = 0.8533
Septoria_leaf_spot: Weight = 0.8154
Spider_mites Two-spotted_spider_mite: Weight = 1.3452
Target_Spot: Weight = 1.2863
Tomato_Yellow_Leaf_Curl_Virus: Weight = 1.1526
Tomato_mosaic_virus: Weight = 1.0915
healthy: Weight = 0.7703
powdery_mildew: Weight = 2.3407


In [ ]:
import os
import tensorflow as tf
from tqdm.keras import TqdmCallback

checkpoint_dir = r"/content/drive/MyDrive/plant_disease_dataset/checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)
checkpoint_path = os.path.join(checkpoint_dir, "best_drone_model.keras")

print("="*60)
print("PHASE 1, 2 & 3: FULL DATASET TRAINING INITIATED")
print("="*60)

# 1. Compile Model (Using the fix we applied earlier)
drone_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=tf.keras.losses.CategoricalFocalCrossentropy(alpha=0.25, gamma=2.0),
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 2. Define Callbacks (Phase 3 Regularization)
checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_path,
    monitor='val_loss',
    save_best_only=False, # Changed to False for debugging
    verbose=1 # Changed to 1 for debugging output
)

early_stopping_cb = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=8, # Wait 8 epochs without improvement before stopping
    restore_best_weights=True
)

# NEW: Dynamic Learning Rate Scheduler
reduce_lr_cb = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,      # Cut learning rate in half
    patience=3,      # If no improvement for 3 epochs
    min_lr=1e-6,     # Don't go below this rate
    verbose=1
)

tqdm_callback = TqdmCallback(verbose=1)

# 3. Start Training Loop (Phase 1 & 2 Applied)
# Note: We are now using the FULL train_ds and passing the class_weight_dict
epochs = 20 # Lowered to 20 since CPU training on 25k images takes longer

history = drone_model.fit(
    train_ds,                      # PHASE 1: Full training data
    validation_data=val_ds,        # PHASE 1: Full validation data
    epochs=epochs,
    class_weight=class_weight_dict, # PHASE 2: Apply calculated penalties
    verbose=0,
    callbacks=[
        checkpoint_cb,
        early_stopping_cb,
        reduce_lr_cb,              # PHASE 3: Dynamic LR
        tqdm_callback
    ]
)

PHASE 1, 2 & 3: FULL DATASET TRAINING INITIATED


0epoch [00:00, ?epoch/s]

0batch [00:00, ?batch/s]

INPUT TO PATCHES : (None, 55, 55, 128)
PATCH TENSOR : (None, 11, 11, 3200)
PATCHES AFTER RESHAPE : (None, None, 3200)
INPUT PATCHES : (None, None, 3200)
NUMBER OF POSITIONS : (121,)
PROJECTED PATCHES : (None, None, 32)
POSITION EMBEDDING : (121, 32)
FINAL OUTPUT : (None, 121, 32)
INPUT TO PATCHES : (None, 55, 55, 128)
PATCH TENSOR : (None, 11, 11, 3200)
PATCHES AFTER RESHAPE : (None, None, 3200)
INPUT PATCHES : (None, None, 3200)
NUMBER OF POSITIONS : (121,)
PROJECTED PATCHES : (None, None, 32)
POSITION EMBEDDING : (121, 32)
FINAL OUTPUT : (None, 121, 32)
INPUT TO PATCHES : (None, 55, 55, 128)
PATCH TENSOR : (None, 11, 11, 3200)
PATCHES AFTER RESHAPE : (None, None, 3200)
INPUT PATCHES : (None, None, 3200)
NUMBER OF POSITIONS : (121,)
PROJECTED PATCHES : (None, None, 32)
POSITION EMBEDDING : (121, 32)
FINAL OUTPUT : (None, 121, 32)

Epoch 1: saving model to /content/drive/MyDrive/plant_disease_dataset/checkpoints/best_drone_model.keras

Epoch 1: finished saving model to /content/drive/M

In [ ]:
print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))

if tf.test.is_built_with_cuda():
    print("TensorFlow is built with CUDA (GPU support).")
else:
    print("TensorFlow is NOT built with CUDA (GPU support). Please check your TensorFlow installation or Colab runtime type.")

# List all physical devices (CPU and GPU)
print("\nPhysical devices available:")
for device in tf.config.list_physical_devices():
    print(device.name, device.device_type)

# Check if a GPU is available and set as the default device
if tf.config.list_physical_devices('GPU'):
    print("\nGPU is available and should be used by TensorFlow Keras operations.")
else:
    print("\nNo GPU found. TensorFlow will run on CPU. Consider changing Colab runtime type to GPU.")

Num GPUs Available:  1
TensorFlow is built with CUDA (GPU support).

Physical devices available:
/physical_device:CPU:0 CPU
/physical_device:GPU:0 GPU

GPU is available and should be used by TensorFlow Keras operations.


If the output indicates that a GPU is available, then your `drone_model.fit()` call will automatically leverage it. You would typically only need to explicitly specify a device (`tf.device('/gpu:0')`) for custom operations or if you're working with multi-GPU setups where you want to control device assignment.

In [ ]:
drone_model = Model(inputs=inputs, outputs=predictions)

In [ ]:
print("="*60)
print("x_inc_out shape :", x_inc_out.shape)

patches = Patches(patch_size)(x_inc_out)
print("patches shape :", patches.shape)

encoded = PatchEncoder(num_patches, projection_dim)(patches)
print("encoded shape :", encoded.shape)

print("="*60)

x_inc_out shape : (None, 28, 28, 128)
INPUT TO PATCHES : (None, 28, 28, 128)
PATCH TENSOR : (None, 5, 5, 3200)
PATCHES AFTER RESHAPE : (None, None, 3200)
patches shape : (None, None, 3200)
INPUT PATCHES : (None, None, 3200)
NUMBER OF POSITIONS : (121,)
PROJECTED PATCHES : (None, None, 32)
POSITION EMBEDDING : (121, 32)
FINAL OUTPUT : (None, 121, 32)
INPUT PATCHES : (None, None, 3200)
NUMBER OF POSITIONS : (121,)
PROJECTED PATCHES : (None, None, 32)
POSITION EMBEDDING : (121, 32)
FINAL OUTPUT : (None, 121, 32)
encoded shape : (None, 121, 32)
